# Injection-recovery with the 21 HARPS-N offsets fitted with the planet

**Setup.** Residuals HARPS-N − SunSim (`rv['res']`) and HARPS-N alone (`rv['obs']`). The planet is injected in the
RVs at the HARPS-N days before any correction. At every trial period the GLS fits together: **one offset per HARPS-N
intervention interval (21)**, a **long-period nuisance sinusoid (≈860 d)** and the planet sinusoid (weighted least
squares, equal weights). Detection: highest peak within 10 % of the injected period and above the **empirical 0.1 %
FAP** (noise-only versions of the series shifted cyclically in time, which keeps the red noise).

**Why 21 offsets and not the BIC-merged 8.** `compare_rv` removes the median of each of the 21 intervals from
`rv['res']`. With the 8 merged offsets, the noise therefore had 21 offsets removed but the injected planet only 8:
the planet was treated more gently than the noise, which made the 8-offset limits too optimistic. With the same 21
offsets for the noise and the planet, both are treated the same way, as they would be in real data. (Redone on
obs − sim without the 21 medians: 21 offsets 0.20–0.24 m/s, BIC-merged 9 offsets 0.28–0.36, the 2 offsets of
Dumusque et al. 2026 0.32–0.39, at 50 % up to 1 yr.)

**The ≈860 d signal.** Dumusque et al. (2026) find signals left at ≈895 and ≈585 d in the HARPS-N solar RVs, and the
teams of the PLATO RV challenge (Cretignier et al. 2026) find spurious long-period signals in the same data: it is in
the data, so it is fitted as a nuisance term.

Sections: (1) injection-recovery on the daily series and the figure for the paper; (2) realistic stellar cadence
(referee); (3) the detection criteria of Zhao et al. (2024), Colwell et al. (2025), Gómez-Vargas et al. (2026) and
Liang et al. (2026).

In [ ]:
import os
import warnings

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from matplotlib.lines import Line2D
import cmasher as cmr

from sunsim import local_path, observations as obs
from sunsim.residuals import ResidualAnalysis

SIM_RV = os.path.join(local_path('personal_data'), 'rv-15.txt')
ORIGIN = '2015-07-18'                # t = 0 of the simulation
FIRST, LAST = '2015-07-18', '2024-01-28'
FIGURES = "/Users/sophiestucki/Documents/rv_opt/figures/final/"   # None: figures not saved

PERIODS = np.sort(np.append(np.linspace(5, 500, 10), 365.))          # injected periods [days], + 1 yr
AMPLITUDES = np.round(np.arange(0.10, 0.701, 0.02), 3)               # injected semi-amplitudes [m/s]
N_PHASES = 100                       # times of transit per (P, K)
FAP = 1e-3                           # detection threshold (empirical)
PERIOD_TOL = 0.1                     # recovered period within 10 % of the injected one
ROTATION = 26                        # [days]
N_NULL = 2000                        # noise-only versions for the empirical FAP
SEED = 1
ADD_FIBRE = False                    # True: also an offset at the fibre change (19 Jan 2018, Dumusque et al. 2026),
                                     # which is not in the HARPS-N intervention list (closest cut: day 850)
RUN_CADENCE = True                   # section 2
RUN_LITERATURE = True                # section 3

M_EARTH_1YR = (2 * np.pi * 6.674e-11 / (365.25 * 86400)) ** (1 / 3) * 5.972e24 / 1.989e30 ** (2 / 3)   # 0.0895 m/s
CMAP = cmr.get_sub_cmap('cmr.sunburst_r', 0.0, 0.8)                 # light: not recovered, dark red: 100 %
INK = '#222222'
HALO = [pe.withStroke(linewidth=3.2, foreground='white')]
LABEL = {'res': 'HARPS-N $-$ SunSim', 'obs': 'HARPS-N'}
PAPER_RC = {'font.size': 8, 'axes.labelsize': 8, 'xtick.labelsize': 7, 'ytick.labelsize': 7, 'legend.fontsize': 6.5}


def save(fig, name):
    if FIGURES:
        fig.savefig(os.path.join(FIGURES, name + '.pdf'), bbox_inches='tight', dpi=300)


def k_2earth(P):
    """K [m/s] of M sin i = 2 Earth masses at period P [d] (1 Msun, circular orbit)."""
    return 2 * M_EARTH_1YR * (np.asarray(P) / 365.25) ** (-1 / 3)


def msini(K, P):
    """M sin i [Earth masses] for K [m/s] and P [d] (1 Msun, circular orbit)."""
    return np.asarray(K) / M_EARTH_1YR * (np.asarray(P) / 365.25) ** (1 / 3)

In [ ]:
t_s, rv_s = np.loadtxt(SIM_RV).T
window = (t_s >= obs.jd_of_date(FIRST) - obs.jd_of_date(ORIGIN)) & (t_s < obs.jd_of_date(LAST) - obs.jd_of_date(ORIGIN) + 1)
t_s, rv_s = t_s[window], rv_s[window]
harps = obs.load_harpsn(local_path('harpsn_sun'), ORIGIN, end=LAST, t_start=FIRST)
cuts = obs.load_interventions(local_path('harpsn_interventions'), ORIGIN)
rv = obs.compare_rv(harps, t_s, rv_s, interventions=cuts)
obs.print_summary(rv=rv)

## Model and tools

In [ ]:
class NuisanceResidualAnalysis(ResidualAnalysis):
    """GLS with nuisance terms fitted together with the sinusoid at every trial frequency (weighted least squares):
    one offset per interval (groups) and a free sinusoid at every period of nuisance_periods [days].
    power = chi2 reduction relative to the nuisance-only fit."""

    def __init__(self, *args, nuisance_periods=(), **kw):
        super().__init__(*args, **kw)
        g = np.unique(self.groups, return_inverse=True)[1]
        X = np.zeros((self.N, int(g.max()) + 1))
        X[np.arange(self.N), g] = 1
        self.n_groups = X.shape[1]
        self.nuisance_periods = tuple(nuisance_periods)
        cols = [X] + [np.column_stack([np.cos(2 * np.pi * self.th / P), np.sin(2 * np.pi * self.th / P)])
                      for P in self.nuisance_periods]
        self.X = np.hstack(cols)
        self.n_nuisance = self.X.shape[1]
        self._WX = self.X * self.w[:, None]
        self._Hinv = np.linalg.inv(self.X.T @ self._WX)
        nf = len(self.freq)
        CC, SS, CS = np.empty(nf), np.empty(nf), np.empty(nf)
        for sl, cos, sin in self._trig():
            Ac, As = cos @ self._WX, sin @ self._WX
            CC[sl] = (cos * cos) @ self.w - np.einsum('fq,qr,fr->f', Ac, self._Hinv, Ac)
            SS[sl] = (sin * sin) @ self.w - np.einsum('fq,qr,fr->f', As, self._Hinv, As)
            CS[sl] = (cos * sin) @ self.w - np.einsum('fq,qr,fr->f', Ac, self._Hinv, As)
        self._CC, self._SS, self._CS = CC, SS, CS
        self._D = CC * SS - CS ** 2
        self._C, self._S = np.zeros(nf), np.zeros(nf)

    def center(self, y):
        """y minus its weighted least-squares fit by the nuisance terms."""
        y = np.asarray(y, dtype=np.float64)
        return y - self.X @ (self._Hinv @ (self._WX.T @ y))

    def nuisance_amplitudes(self, y=None):
        """Semi-amplitudes of the nuisance sinusoids fitted (with the offsets) to y (default self.y)."""
        y = self.y if y is None else y
        c = self._Hinv @ (self._WX.T @ y)
        return np.hypot(c[self.n_groups::2], c[self.n_groups + 1::2])

    def _gls(self, y):
        y = np.asarray(y, dtype=np.float64)
        one = y.ndim == 1
        y = y[:, None] if one else y
        wy = self.center(y)
        YY = self.w @ wy ** 2
        wy = wy * self.w[:, None]
        YC, YS = np.empty((len(self.freq), y.shape[1])), np.empty((len(self.freq), y.shape[1]))
        for sl, cos, sin in self._trig():
            YC[sl], YS[sl] = cos @ wy, sin @ wy
        CC, SS, CS, D = (v[:, None] for v in (self._CC, self._SS, self._CS, self._D))
        a = (YC * SS - YS * CS) / D
        b = (YS * CC - YC * CS) / D
        power = (SS * YC * YC + CC * YS * YS - 2 * CS * YC * YS) / (YY * D)
        out = (power, a, b, np.zeros_like(a))
        return tuple(v[:, 0] for v in out) if one else out

    def fap(self, power):
        prob = (1 - np.asarray(power, dtype=np.float64)) ** ((self.N - self.n_nuisance - 2) / 2)
        Mprob = self.M * prob
        out = np.where(Mprob > 0.01, 1 - (1 - prob) ** self.M, Mprob)
        return out if out.ndim else float(out)

    def power_level(self, fap):
        prob = 1 - (1 - np.asarray(fap, dtype=np.float64)) ** (1 / self.M)
        return 1 - prob ** (2 / (self.N - self.n_nuisance - 2))

    def subset(self, keep, **kw):
        """The same model on the points keep (boolean or indices), e.g. a shorter time span or a sub-sample."""
        kw.setdefault('fbeg', self.fbeg)
        kw.setdefault('fend', self.fend)
        return NuisanceResidualAnalysis(self.t[keep], self.y[keep], groups=self.groups[keep],
                                        t_planet=self.t_planet[keep], nuisance_periods=self.nuisance_periods, **kw)


def null_peaks(ra, n=N_NULL, seed=SEED, chunk=250):
    """Highest power and its period for n noise-only versions of ra.y, shifted cyclically by a random number of points
    (at least 5 % of N away from no shift)."""
    rng = np.random.default_rng(seed)
    k = rng.integers(int(0.05 * ra.N), int(0.95 * ra.N) + 1, size=n)
    pmax, P = np.empty(n), np.empty(n)
    for i in range(0, n, chunk):
        pg = ra.periodogram(np.stack([np.roll(ra.y, s) for s in k[i:i + chunk]], axis=1))
        pmax[i:i + chunk], P[i:i + chunk] = pg['pmax'], pg['P']
    return {'pmax': pmax, 'P': P}


def empirical_level(null, fap=FAP):
    """Power above which the highest peak of a fraction fap of the noise-only versions lies."""
    return np.quantile(null['pmax'], 1 - fap)


def limits(ra, injection, null, levels=(50, 100)):
    """Recovery rate [%] (n_amplitudes, n_periods) and the K reached at every level (NaN: not within the grid).
    100 %: smallest K above which every K of the grid is 100 % recovered."""
    fap = ra.fap(empirical_level(null))
    rate = ra.detection_rate(fap=fap, period_tol=PERIOD_TOL, injection=injection)
    K = injection['amplitudes']
    out = {}
    for level in levels:
        lim = np.full(rate.shape[1], np.nan)
        for j in range(rate.shape[1]):
            if level == 100:
                bad = np.flatnonzero(rate[:, j] < 100)
                lim[j] = K[0] if len(bad) == 0 else (K[bad[-1] + 1] if bad[-1] < len(K) - 1 else np.nan)
            else:
                above = np.flatnonzero(rate[:, j] >= level)
                if len(above):
                    i = above[0]
                    lim[j] = K[0] if i == 0 else np.interp(level, rate[i - 1:i + 1, j], K[i - 1:i + 1])
        out[level] = lim
    return rate, out

### The model: 21 offsets + the long-period nuisance sinusoid
The nuisance period is the highest peak of the residuals once the 21 offsets are fitted.

In [ ]:
offsets_only = NuisanceResidualAnalysis.from_comparison(rv, 'res')                 # 21 offsets, time order
GROUPS = offsets_only.groups
if ADD_FIBRE:
    GROUPS = GROUPS + (offsets_only.t >= obs.jd_of_date('2018-01-19') - obs.jd_of_date(ORIGIN))
    offsets_only = NuisanceResidualAnalysis(offsets_only.t, offsets_only.y, groups=GROUPS, t_planet=offsets_only.t_planet)
P_NUISANCE = offsets_only.periodogram()['P']

models = {}
for key, kw in (('res', {}), ('obs', {'period_medians': True})):
    b = NuisanceResidualAnalysis.from_comparison(rv, key, **kw)
    assert np.array_equal(b.t, offsets_only.t)
    models[key] = NuisanceResidualAnalysis(b.t, b.y, groups=GROUPS, t_planet=b.t_planet,
                                           nuisance_periods=(P_NUISANCE,),
                                           name='%s, %d offsets + %.0f d' % (LABEL[key].replace('$-$', '-'),
                                                                              offsets_only.n_groups, P_NUISANCE))
    print('%-45s %d points, nuisance semi-amplitude %.2f m/s'
          % (models[key].name, models[key].N, models[key].nuisance_amplitudes()[0]))

## 1. Injection-recovery on the daily series

In [ ]:
inj, nulls, rates, lims = {}, {}, {}, {}
for key, ra in models.items():
    inj[key] = ra.inject(PERIODS, AMPLITUDES, n_phases=N_PHASES, fap=FAP, fit=False, before=True, verbose=False)
    nulls[key] = null_peaks(ra)
    rates[key], lims[key] = limits(ra, inj[key], nulls[key])
    pg = ra.periodogram()
    print('%-45s highest peak of the series itself: %.0f d, empirical FAP %.3f'
          % (ra.name, pg['P'], np.mean(nulls[key]['pmax'] >= pg['pmax'])))

i1 = int(np.argmin(np.abs(PERIODS - 365)))
print('\n%-45s' % 'K [m/s] at 50 % / 100 %, P =' + ''.join('%11.0f' % p for p in PERIODS))
for key, ra in models.items():
    print('%-45s' % ra.name + ''.join('%5.2f/%5.2f' % (a, b) for a, b in zip(lims[key][50], lims[key][100])))
print('\nM sin i at 1 yr [Earth masses], 50 %% / 100 %%: %.1f / %.1f'
      % (msini(lims['res'][50][i1], 365.25), msini(lims['res'][100][i1], 365.25)))

# false alarms: noise-only versions that the criterion counts as a detection at an injected period
fa = [100 * np.mean((nulls['res']['pmax'] >= empirical_level(nulls['res']))
                    & (np.abs(nulls['res']['P'] - P) / P <= PERIOD_TOL)) for P in PERIODS]
print('false alarms of the criterion [%]:', ' '.join('%.0f d %.2f' % (p, f) for p, f in zip(PERIODS, fa)))

### Figure for the paper
(a) % of the injections recovered in HARPS-N − SunSim, (b) in HARPS-N, (c) K recovered at 50, 70, 90 and 100 % against
the period (solid: HARPS-N − SunSim, dashed: HARPS-N; missing: not reached within the grid). Thick line:
M sin i = 2 M$_\oplus$.

In [ ]:
LEVELS = (50, 70, 90, 100)
with plt.rc_context(PAPER_RC):
    fig = plt.figure(figsize=(7.1, 2.5))
    gs = fig.add_gridspec(1, 5, width_ratios=[1, 1, 0.045, 0.30, 1], wspace=0.06)
    axa = fig.add_subplot(gs[0, 0])
    axb = fig.add_subplot(gs[0, 1], sharex=axa, sharey=axa)
    cax = fig.add_subplot(gs[0, 2])
    axc = fig.add_subplot(gs[0, 4], sharex=axa, sharey=axa)
    for ax, key in ((axa, 'res'), (axb, 'obs')):
        im = ax.pcolormesh(PERIODS, AMPLITUDES, rates[key], shading='nearest', cmap=CMAP, vmin=0, vmax=100,
                           rasterized=True)
    fig.colorbar(im, cax=cax)
    cax.set_title('[%]', fontsize=7, pad=4)
    x0, x1 = axa.get_xlim()
    y0, y1 = axa.get_ylim()
    pp = np.linspace(max(x0, 20), x1, 300)
    for key, ls in (('res', '-'), ('obs', '--')):
        _, lv = limits(models[key], inj[key], nulls[key], levels=LEVELS)
        for level, mk in zip(LEVELS, 'osD^'):
            axc.plot(PERIODS, lv[level], ls, marker=mk, color=CMAP(level / 100), lw=1.3, ms=3, zorder=4,
                     mfc='white' if key == 'obs' else None, label='%d %%' % level if key == 'res' else None)
    for ax, tag, title in ((axa, '(a)', LABEL['res']), (axb, '(b)', LABEL['obs']), (axc, '(c)', None)):
        ax.plot(pp, k_2earth(pp), color=INK, lw=1.8, path_effects=HALO, zorder=3)
        ax.axvline(365.25, color='0.45', lw=0.7, ls=':', zorder=2)
        ax.set_xlim(x0, x1)
        ax.set_ylim(y0, y1)
        ax.set_xlabel('Injected period [d]')
        ax.text(0.03, 0.97, tag + ('  ' + title if title else ''), transform=ax.transAxes, ha='left', va='top',
                path_effects=HALO)
    axa.text(x1 - 10, k_2earth(x1 - 10) - 0.012, '2 M$_\\oplus$', color=INK, ha='right', va='top', fontsize=7.5,
             fontweight='bold', path_effects=HALO, zorder=4)
    axb.text(372, y1 - 0.01, '1 yr', fontsize=6.5, color='0.35', va='top')
    axa.set_ylabel('Semi-amplitude $K$ [m s$^{-1}$]')
    plt.setp(axb.get_yticklabels(), visible=False)
    axc.legend(title='Recovered', loc='upper left', bbox_to_anchor=(0.0, 0.9), ncol=2, frameon=False, title_fontsize=6.5,
               handlelength=1.5, columnspacing=0.8)
save(fig, 'injection_recovery')
plt.show()

## 2. Realistic cadence for another star (referee)
A Sun-like star is observed once per night, only part of the year. Same test on sub-sampled versions of the daily
series: a season of 8 months per year (its centre drawn at random) and 150 or 100 random nights per season;
N_DRAWS random sub-samples per cadence. EXTRA_NOISE [m/s]: white noise added to every point (0: the daily solar values
as they are; a single stellar exposure does not average the oscillations and granulation over the day, so ~0.3 m/s is a
more conservative test). Same frequency grid as the daily series; the ≈860 d period is kept.

In [ ]:
CADENCES = {'150 nights / 8-month season': (150, 240),          # name: (nights per season, season length [d])
            '100 nights / 8-month season': (100, 240)}
N_DRAWS = 3
EXTRA_NOISE = 0.0
AMPLITUDES_SUB = np.round(np.arange(0.10, 1.001, 0.05), 3)
N_PHASES_SUB = 50


def subsample(ra, n_nights, season, rng):
    """Indices of ra kept for a season of `season` days per year (random centre) with n_nights random nights per
    season (all of them if fewer). Returns the indices and the number of seasons."""
    centre = rng.uniform(0, 365.25)
    d = np.mod(ra.t - centre + 365.25 / 2, 365.25) - 365.25 / 2
    in_season = np.abs(d) <= season / 2
    year = np.floor((ra.t - centre + 365.25 / 2) / 365.25)
    keep = []
    for y in np.unique(year[in_season]):
        idx = np.flatnonzero(in_season & (year == y))
        keep.append(np.sort(rng.choice(idx, size=min(n_nights, len(idx)), replace=False)))
    return np.concatenate(keep), len(keep)


if RUN_CADENCE:
    rng = np.random.default_rng(SEED)
    cadence = {}                                                   # (key, cadence) -> list of (limits, N)
    for cname, (n_nights, season) in CADENCES.items():
        for d in range(N_DRAWS):
            keep, n_seasons = subsample(models['res'], n_nights, season, rng)     # same nights for res and obs
            noise = np.random.default_rng(rng.integers(1 << 31)).normal(0, EXTRA_NOISE, len(keep)) if EXTRA_NOISE else 0
            for key in ('res', 'obs'):
                ra = models[key].subset(keep, name='%s, %s' % (key, cname))
                ra.y = ra.y + noise
                inj_ = ra.inject(PERIODS, AMPLITUDES_SUB, n_phases=N_PHASES_SUB, fap=FAP, fit=False, before=True,
                                 verbose=False)
                cadence.setdefault((key, cname), []).append((limits(ra, inj_, null_peaks(ra))[1], ra.N))
            print('%s, draw %d: %d points, %d seasons' % (cname, d + 1, len(keep), n_seasons), flush=True)

In [ ]:
if RUN_CADENCE:
    def fmt(v):
        v = np.asarray(v, dtype=float)
        if np.isnan(v).all():
            return '-'
        r = '%.2f-%.2f' % (np.nanmin(v), np.nanmax(v)) if np.isfinite(v).all() else '%d/%d' % (np.isfinite(v).sum(), len(v))
        return '%.2f [%s]' % (np.nanmedian(v), r)

    # K [m/s]: median of the draws [min-max] ('1/3': reached in 1 of the 3 draws); '-': not reached within the grid
    for key in ('res', 'obs'):
        for level in (50, 100):
            print('\n%s, K [m/s] recovered at %d %%' % (LABEL[key].replace('$-$', '-'), level))
            print('%8s %18s' % ('P [d]', 'daily (%d)' % models[key].N) + ''.join(
                '%22s' % ('%s (%d)' % (c.split(' /')[0], np.median([r[1] for r in cadence[key, c]]))) for c in CADENCES))
            for j, P in enumerate(PERIODS):
                v = lims[key][level][j]
                print('%8.0f %18s' % (P, '-' if np.isnan(v) else '%.2f' % v) + ''.join(
                    '%22s' % fmt([r[0][level][j] for r in cadence[key, c]]) for c in CADENCES))
    print('\nM sin i at 1 yr, 50 % [Earth masses]: ' + ', '.join(
        '%s %.1f' % (c, msini(np.nanmedian([r[0][50][i1] for r in cadence['res', c]]), 365.25)) for c in CADENCES))

### Figure: realistic cadence
K recovered at 50 % (a) and 100 % (b): daily series (solid) and sub-sampled series (median of the draws, shaded:
min–max of the draws). Dark red: HARPS-N − SunSim, orange: HARPS-N. Thick line: M sin i = 2 M$_\oplus$.

In [ ]:
if RUN_CADENCE:
    COLOR = {'res': CMAP(0.92), 'obs': CMAP(0.42)}
    cases = [('daily', '-', 'o', 'daily')] + [(c, ls, mk, c.split(' /')[0] + ' / season')
                                              for c, ls, mk in zip(CADENCES, ('--', ':'), ('s', 'D'))]
    with plt.rc_context(PAPER_RC), warnings.catch_warnings():
        warnings.simplefilter('ignore', RuntimeWarning)                 # all-NaN medians
        fig, axes = plt.subplots(1, 2, figsize=(7.1, 2.7), sharex=True, sharey=True, gridspec_kw={'wspace': 0.06})
        for ax, level, tag in zip(axes, (50, 100), ('(a)  50 % recovered', '(b)  100 % recovered')):
            for key in ('obs', 'res'):
                for name, ls, mk, _ in cases:
                    if name == 'daily':
                        med = lims[key][level]
                    else:
                        L = np.array([r[0][level] for r in cadence[key, name]])
                        med = np.nanmedian(L, axis=0)
                        ax.fill_between(PERIODS, np.nanmin(L, axis=0), np.nanmax(L, axis=0), color=COLOR[key],
                                        alpha=0.08, lw=0, zorder=1)
                    ax.plot(PERIODS, med, ls, marker=mk, color=COLOR[key], lw=1.3, ms=3, zorder=4,
                            mfc='white' if name != 'daily' else COLOR[key])
            pp = np.linspace(20, 520, 300)
            ax.plot(pp, k_2earth(pp), color=INK, lw=1.8, path_effects=HALO, zorder=3)
            ax.axvline(365.25, color='0.45', lw=0.7, ls=':', zorder=2)
            ax.text(0.03, 0.97, tag, transform=ax.transAxes, ha='left', va='top', path_effects=HALO)
            ax.set_xlabel('Injected period [d]')
        axes[0].set_xlim(0, 520)
        axes[0].set_ylim(0.1, 1.0)
        axes[0].set_ylabel('Semi-amplitude $K$ [m s$^{-1}$]')
        axes[0].text(515, k_2earth(515) + 0.012, '2 M$_\\oplus$', color=INK, ha='right', va='bottom', fontsize=7.5,
                     fontweight='bold', path_effects=HALO, zorder=5)
        axes[0].text(372, 0.11, '1 yr', fontsize=6.5, color='0.35', va='bottom')
        h1 = [Line2D([], [], color=COLOR[k], lw=1.3, label=LABEL[k]) for k in ('res', 'obs')]
        h2 = [Line2D([], [], color='0.3', ls=ls, marker=mk, ms=3, lw=1.1, mfc='white' if n != 'daily' else '0.3',
                     label=lab) for n, ls, mk, lab in cases]
        fig.legend(handles=h1 + h2, loc='upper center', bbox_to_anchor=(0.5, -0.07), ncol=5, frameon=False,
                   handlelength=2.4)
    save(fig, 'injection_cadence')
    plt.show()

## 3. The criteria of the other studies

| | data | injections | detection |
|---|---|---|---|
| Zhao et al. (2024) | HARPS-N 2015–2024, daily | P log-spaced 10–550 d, K 0.1–0.5 m/s (step 0.033), one injection per (P, K) | power at the injected period > 0.1 % FAP ("numerically estimated"); limit = lowest K detected |
| Colwell et al. (2025) | HARPS-N 2015–2018 (3 yr), exposures | P = 50 d, K = 0.1–1.0 m/s; P = 10–250 d at K = 1 m/s | period of the highest peak and amplitude fitted, compared with the injected ones (no FAP) |
| Gómez-Vargas et al. (2026) | HARPS-N 2015–2024, daily | P 10–550 d, 10 realisations | power within ±5 % of the injected period > 0.1 % FAP in ≥ 70 % of the realisations |
| Liang et al. (2026) | NEID 2020–2024, 1294 d | 500 trials, P log-uniform 2.5–400 d, K uniform 0.1–0.7 m/s, random phase | highest candidate, threshold raised until no false positive among the 500 trials; recovered if its period is within 20 % and \|K_rec − K\|/K < 1 |

Choices where the papers do not say: the 0.1 % FAP of Zhao is a bootstrap (values shuffled within the intervals), the
period tolerance of Zhao is ±5 % (as Gómez-Vargas), the FAP of Gómez-Vargas is analytic, the success of Colwell is the
highest peak within 5 % of the period and the amplitude within 20 %. Same model as above (21 offsets + nuisance).
Periods where the series without any planet already passes the criterion are marked * (not usable).

In [ ]:
def injected(ra, P, K, t0):
    """ra.y + K sin(2 pi (t_planet - t0) / P), one column per (P, K, t0) (arrays of the same length or scalars)."""
    P, K, t0 = np.broadcast_arrays(np.atleast_1d(P), np.atleast_1d(K), np.atleast_1d(t0))
    return ra.y[:, None] + K[None] * np.sin(2 * np.pi * (ra.t_planet[:, None] - t0[None]) / P[None])


def analyse(ra, Y, P_inj, tol=0.05, chunk=200):
    """For every column of Y: period, power and semi-amplitude of the highest peak, and the highest power within
    ±tol of the injected period P_inj (one per column)."""
    P_inj = np.broadcast_to(np.atleast_1d(P_inj), (Y.shape[1],))
    out = {k: np.empty(Y.shape[1]) for k in ('P', 'pmax', 'K', 'local')}
    for i in range(0, Y.shape[1], chunk):
        sl = slice(i, i + chunk)
        power, a, b, _ = ra._gls(Y[:, sl])
        k = power.argmax(axis=0)
        c = np.arange(power.shape[1])
        out['P'][sl], out['pmax'][sl], out['K'][sl] = 1 / ra.freq[k], power[k, c], np.hypot(a[k, c], b[k, c])
        f = ra.freq[:, None]
        near = (f >= 1 / ((1 + tol) * P_inj[sl])) & (f <= 1 / ((1 - tol) * P_inj[sl]))
        out['local'][sl] = np.where(near, power, -np.inf).max(axis=0)
    return out


def shuffle_level(ra, fap=FAP, n=N_NULL, seed=SEED):
    """Bootstrap: power of the highest peak exceeded by a fraction fap of the versions with the values permuted
    inside every interval."""
    rng = np.random.default_rng(seed)
    Y = np.repeat(ra.y[:, None], n, axis=1)
    for g in np.unique(ra.groups):
        idx = np.flatnonzero(ra.groups == g)
        Y[idx] = rng.permuted(Y[idx], axis=0)
    return np.quantile(analyse(ra, Y, np.ones(n))['pmax'], 1 - fap)


def limit(K, rate, level):
    """Lowest K where rate [%] (along K) first reaches level (linear interpolation; NaN: never)."""
    above = np.flatnonzero(rate >= level)
    if len(above) == 0:
        return np.nan
    i = above[0]
    return K[0] if i == 0 else float(np.interp(level, rate[i - 1:i + 1], K[i - 1:i + 1]))


summary = {}

### Zhao et al. (2024)
Their grid; power within ±5 % of the injected period above the bootstrap 0.1 % FAP. Their one injection per (P, K) is
replaced by 100 phases: their "lowest K detected" corresponds to ~50 %.

In [ ]:
if RUN_LITERATURE:
    P_Z = np.geomspace(10, 550, 12)
    K_Z = np.arange(0.1, 0.5 + 1e-9, 0.1 / 3)
    rate_Z, fp_Z = {}, {}
    for key, ra in models.items():
        thr = shuffle_level(ra)
        fp_Z[key] = np.array([analyse(ra, ra.y[:, None], P)['local'][0] >= thr for P in P_Z])
        rate = np.empty((len(K_Z), len(P_Z)))
        for j, P in enumerate(P_Z):
            t0 = ra.t[0] + P * np.arange(N_PHASES) / N_PHASES
            for i, K in enumerate(K_Z):
                rate[i, j] = 100 * np.mean(analyse(ra, injected(ra, P, K, t0), P)['local'] >= thr)
        rate_Z[key] = rate
        lim = [limit(K_Z, rate[:, j], 50) for j in range(len(P_Z))]
        print('%-45s K at 50 %%:' % ra.name)
        print('   ' + ' '.join('%4.0f d %.2f%s' % (p, k, '*' if f else ' ') for p, k, f in zip(P_Z, lim, fp_Z[key])))
    lz = np.array([limit(K_Z, rate_Z['res'][:, j], 50) for j in range(len(P_Z))])[~fp_Z['res']]
    summary['Zhao et al. (2024)'] = ('0.18 m/s (10-550 d)', '%.2f-%.2f m/s (%d of %d periods usable)'
                                     % (np.nanmin(lz), np.nanmax(lz), len(lz), len(P_Z)))

### Colwell et al. (2025)
First 3 years (2015-07-18 to 2018-07-18). Amplitude sweep at P = 50 d and period sweep at K = 1 m/s. Recovered:
highest peak within 5 % of the period and fitted K within 20 %. Their result: 0.2 m/s at 50 d.

In [ ]:
if RUN_LITERATURE:
    t_end = obs.jd_of_date('2018-07-18') - obs.jd_of_date(ORIGIN)
    ra_c = models['res'].subset(models['res'].t <= t_end, fbeg=None, fend=None, name='HARPS-N - SunSim, 2015-2018')
    print('%d days, %d offsets' % (ra_c.N, ra_c.n_groups))
    K_C = np.arange(0.1, 1.0 + 1e-9, 0.1)
    t0 = ra_c.t[0] + 50 * np.arange(N_PHASES) / N_PHASES
    print('P = 50 d:   K    recovered [%]   median |dK|/K   median |dP|/P')
    rec_C = []
    for K in K_C:
        r = analyse(ra_c, injected(ra_c, 50.0, K, t0), 50.0)
        ok = (np.abs(r['P'] - 50) / 50 <= 0.05) & (np.abs(r['K'] - K) / K <= 0.2)
        rec_C.append(100 * ok.mean())
        print('          %4.1f  %8.0f        %10.3f      %10.4f' % (K, 100 * ok.mean(), np.median(np.abs(r['K'] - K) / K),
                                                                np.median(np.abs(r['P'] - 50) / 50)))
    print('P sweep at K = 1 m/s:', end='')
    for P in (10, 25, 40, 50, 75, 100, 150, 200, 250):
        r = analyse(ra_c, injected(ra_c, float(P), 1.0, ra_c.t[0] + P * np.arange(N_PHASES) / N_PHASES), float(P))
        print('  %d d %.0f %%' % (P, 100 * np.mean(np.abs(r['P'] - P) / P <= 0.05)), end='')
    print()
    summary['Colwell et al. (2025)'] = ('0.2 m/s at 50 d', '%.2f m/s at 50 d (50 %% recovered)'
                                        % limit(K_C, np.array(rec_C), 50))

### Gómez-Vargas et al. (2026)
P 10–550 d, 10 realisations (random phases) per (P, K), power within ±5 % of the injected period above the analytic
0.1 % FAP in ≥ 70 % of them.

In [ ]:
if RUN_LITERATURE:
    rng = np.random.default_rng(SEED)
    K_G = np.arange(0.05, 0.5 + 1e-9, 0.025)
    rate_G, fp_G = {}, {}
    for key, ra in models.items():
        thr = ra.power_level(FAP)
        fp_G[key] = np.array([analyse(ra, ra.y[:, None], P)['local'][0] >= thr for P in P_Z])
        rate = np.empty((len(K_G), len(P_Z)))
        for j, P in enumerate(P_Z):
            t0 = ra.t[0] + P * rng.random(10)
            for i, K in enumerate(K_G):
                rate[i, j] = 100 * np.mean(analyse(ra, injected(ra, P, K, t0), P)['local'] >= thr)
        rate_G[key] = rate
        lim = [limit(K_G, rate[:, j], 70) for j in range(len(P_Z))]
        print('%-45s K at 70 %%:' % ra.name)
        print('   ' + ' '.join('%4.0f d %.2f%s' % (p, k, '*' if f else ' ') for p, k, f in zip(P_Z, lim, fp_G[key])))
    lg = np.array([limit(K_G, rate_G['res'][:, j], 70) for j in range(len(P_Z))])[~fp_G['res']]
    summary['Gómez-Vargas et al. (2026)'] = ('>= 0.25 m/s (10-550 d)', '%.2f-%.2f m/s (%d of %d periods usable)'
                                             % (np.nanmin(lg), np.nanmax(lg), len(lg), len(P_Z)))

### Liang et al. (2026)
500 trials (P log-uniform 2.5–400 d, K uniform 0.1–0.7 m/s, random phase). The highest peak of each trial is accepted
if its power exceeds a threshold; a recovery if its period is within 20 % and |K_rec − K|/K < 1, otherwise a false
positive. The threshold is the lowest one with no false positive among the 500 trials. Grid down to 2.2 d.

In [ ]:
if RUN_LITERATURE:
    rng = np.random.default_rng(SEED)
    n_tr = 500
    P_L = np.exp(rng.uniform(np.log(2.5), np.log(400), n_tr))
    K_L = rng.uniform(0.1, 0.7, n_tr)
    ra_l = models['res'].subset(np.arange(models['res'].N), fend=1 / 2.2, name='HARPS-N - SunSim')
    t0 = ra_l.t[0] + P_L * rng.random(n_tr)
    r = analyse(ra_l, injected(ra_l, P_L, K_L, t0), P_L)
    good = (np.abs(r['P'] - P_L) / P_L <= 0.2) & (np.abs(r['K'] - K_L) / K_L < 1)
    thr = r['pmax'][~good].max() if (~good).any() else 0
    found = good & (r['pmax'] > thr)
    print('threshold power %.4f (analytic FAP %.1e)' % (thr, ra_l.fap(thr)))
    print('recovered %d / %d, of which %d with K < 0.3 m/s' % (found.sum(), n_tr, (found & (K_L < 0.3)).sum()))
    bins = np.arange(0.1, 0.71, 0.1)
    for lo, hi in zip(bins[:-1], bins[1:]):
        m = (K_L >= lo) & (K_L < hi)
        print('  K %.1f-%.1f m/s: %3.0f %% recovered (%d trials)' % (lo, hi, 100 * found[m].mean(), m.sum()))
    summary['Liang et al. (2026)'] = ('238 / 500 recovered, 13 with K < 0.3 m/s (NEID, AESTRA)',
                                      '%d / 500 recovered, %d with K < 0.3 m/s' % (found.sum(), (found & (K_L < 0.3)).sum()))

## Summary

In [ ]:
print('Our criterion (empirical FAP, highest peak within 10 %%), %s:' % models['res'].name)
print('  50 %% recovered: %.2f-%.2f m/s up to 1 yr (%.2f at 1 yr, %.1f Earth masses), %.2f-%.2f m/s at 390-500 d'
      % (np.nanmin(lims['res'][50][:i1 + 1]), np.nanmax(lims['res'][50][:i1 + 1]), lims['res'][50][i1],
         msini(lims['res'][50][i1], 365.25), np.nanmin(lims['res'][50][i1 + 1:]), np.nanmax(lims['res'][50][i1 + 1:])))
if RUN_LITERATURE:
    print('\n%-28s %-58s %s' % ('criterion of', 'their result', models['res'].name))
    for name, (theirs, ours) in summary.items():
        print('%-28s %-58s %s' % (name, theirs, ours))